# 随机种子与配置文件笔记

> 目标：理解为什么要做这件事，然后能从零写出 `config.py` 并接入训练代码。

## 一、先看问题：没有配置文件时会发生什么

你现在的训练代码大概长这样：

```python
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(10):
    ...
```

`64`、`1e-3`、`10` 这些值直接写死在代码里，叫做**硬编码**。

带来三个问题：

**问题 1：改参数麻烦**
想把 `batch_size` 从 64 改成 128，要进代码找那一行，改完还容易漏改其他地方。

**问题 2：实验无法对比**
你跑了三组实验，事后想不起来哪组用的什么 `lr`，结果没有可比性。

**问题 3：结果不可复现**
同样的代码跑两次，loss 曲线不一样——因为模型初始化、Dropout、数据打乱都用了随机数，每次随机数不同，结果就不同。

这三个问题，分别由**配置文件**和**随机种子**解决。

## 二、随机种子：让结果可复现

### 2.1 哪些地方用了随机数

PyTorch 训练过程中，有四个地方会用到随机数：

| 随机来源              | 影响                         |
| --------------------- | ---------------------------- |
| 模型参数初始化        | 每次初始值不同，收敛路径不同 |
| Dropout               | 每次丢弃不同的神经元         |
| DataLoader shuffle    | 每个 epoch 数据顺序不同      |
| NumPy / Python random | 数据预处理中的随机操作       |

只要有一个没固定，两次运行结果就会不同。

### 2.2 固定种子的意义

固定种子 = 固定所有随机数序列。
同样的种子 + 同样的代码 → 每次跑出来结果完全一致。

这样你才能判断：**结果的变化是代码改动导致的，还是随机性导致的。**

## 三、配置文件：集中管理超参数

### 3.1 什么是超参数

训练时你需要手动设定、不由模型自己学习的值，叫超参数：

| 超参数       | 含义                           | 典型值 |
| ------------ | ------------------------------ | ------ |
| `seed`       | 随机种子                       | 42     |
| `batch_size` | 每次喂给模型多少条数据         | 64     |
| `lr`         | 学习率，控制每次参数更新的步长 | 1e-3   |
| `num_epochs` | 训练多少轮                     | 10     |

### 3.2 配置文件解决什么

把所有超参数集中放在一个地方：

```
改参数  → 只改 config.py，训练代码不动
做实验  → 每组实验对应一个 config，参数有记录
给别人  → 别人拿到 config 就能复现你的结果
```

## 四、接入训练代码

改动只有三处，其余代码不动：

```python
from config import config, set_seed

# 1.训练开始前的第一行就固定种子
set_seed(config['seed'])

# 2. Dataloader 用config里的batch_size
train_loader=DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=0
)

# 3. optimizer 用config里的 lr
optimizer=torch.optim.Adam(model.parameters(),lr=config['lr'])

# 4. 训练循环，用config里的num_epochs
for epoch in range(config['num_epochs']):
    pass
```

`set_seed` 必须在所有其他操作之前调用，否则在它之前发生的随机操作就没被固定。

## 五、验证是否生效

跑两次训练，对比第一个 epoch 的 loss：

```python
# 如果两次输出完全一样，说明种子生效
# Epoch 1 | Loss: 0.3521 | Acc: 0.8934
# Epoch 1 | Loss: 0.3521 | Acc: 0.8934  ← 完全一致
```

如果两次结果不同，检查 `set_seed` 是否在代码第一行调用。

## 六、完整 config.py 文件

In [ ]:
import torch
import random
import numpy as np

config={
    'seed': 42,
    'batch_size': 64,
    'lr':1e-3,
    'num_epochs': 10
}

def set_seed(seed:int):
    torch.manual_seed(seed)         # 固定 CPU 上的随机数
    torch.cuda.manual_seed(seed)        # 固定 GPU 上的随机数
    random.seed(seed)           # 固定 Python 内置Rand
    np.random.seed(seed)            # 固定 numpy 上的随机数
    torch.backends.cudnn.deterministic=True         # 固定 cuDNN 卷积算法选择

`42` 是惯例种子值，没有特殊含义，用任何整数都行，关键是固定下来。

## 七、一句话总结

| 工具     | 解决的问题                                     |
| -------- | ---------------------------------------------- |
| 随机种子 | 结果不可复现，两次跑出来不一样                 |
| 配置文件 | 超参数散落在代码各处，改参数麻烦，实验无法对比 |

两者合在一起，是所有正式 ML 项目的最基础工程规范。